# non-contiguous tensors and as_strided — procedural drill

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `as-strided-noncontig-source`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five patterns around tensor memory layout that ramp from reading strides → recognizing transpose breaks contiguity → seeing `.view()` fail on non-contig → fixing it with `.contiguous()` → building a zero-copy sliding-window view via `as_strided`. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Applied patterns and advanced` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `as-strided-noncontig-source`**, which bridges to the bank subtopic `Numpy: Applied patterns and advanced` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "as-strided-noncontig-source"
DD_SUBTOPIC = "Numpy: Applied patterns and advanced"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Strides and contiguity — quick refresher

**Stride** = number of elements to skip in storage to advance one step along that axis.
- A contiguous `(H, W)` tensor has stride `(W, 1)`.
- A contiguous `(B, C, H, W)` tensor has stride `(C*H*W, H*W, W, 1)`.

**Contiguity** = the strides match the row-major layout of the current shape.
- `.T` swaps strides but not data → the result is a view but not contiguous.
- `.view()` requires contiguous input — it never copies.
- `.reshape()` makes a view if possible, copies if not.
- `.contiguous()` forces a row-major copy if the tensor isn't already contiguous.

**`as_strided(size, stride)`** is the lowest-level view constructor — you provide the exact shape and stride pair. Bypasses all safety checks; trust the values you pass.

### Exercise 1 — read the strides of a contiguous tensor

> ```yaml
> Difficulty: ⚪⚪⚪⚪⚪
> Bloom level: Remember
> LO: Recall what `.stride()` returns for a contiguous 2-D tensor.
> Keywords: strides, memory-layout, contiguous
> ```

**KCs targeted:** `strides-anatomy`

Implement `ex1_get_strides(x)` to return the strides of `x` as a Python tuple of ints.

For a contiguous `(H, W)` float tensor, the strides are `(W, 1)` — moving one step along axis 0 skips `W` elements (a full row), moving one step along axis 1 skips 1 element (one column).

Just return `tuple(x.stride())`.

In [ ]:
def ex1_get_strides(x: Tensor) -> tuple:
    """Return x.stride() as a tuple of ints."""
    raise NotImplementedError()


def _test_ex1():
    x = t.zeros(3, 5)
    s = ex1_get_strides(x)
    assert isinstance(s, tuple), f'expected tuple, got {type(s)}'
    assert s == (5, 1), f'expected (5, 1) for (3, 5) contiguous tensor, got {s}'

    y = t.zeros(4, 7, 9)
    assert ex1_get_strides(y) == (63, 9, 1), f'expected (63, 9, 1), got {ex1_get_strides(y)}'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_get_strides(x: Tensor) -> tuple:
    return tuple(x.stride())
```

**How to read the result.** For a contiguous tensor of shape `(d0, d1, ..., dn)`, the stride is `(d1*d2*...*dn, d2*...*dn, ..., dn, 1)`. It's just the product of all later dimensions — a recipe for converting an N-D index into a flat memory offset.
</details>

### Exercise 2 — transpose breaks contiguity

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Bloom level: Apply
> LO: Apply `is_contiguous()` to show that `.T` returns a non-contiguous view.
> Keywords: transpose, non-contiguous, view-vs-copy
> ```

**KCs targeted:** `transpose-makes-noncontig`

Implement `ex2_transpose_contig_flags(x)` to return a tuple `(orig_contig, transposed_contig)` — the contiguity flag of the original tensor and of its transpose.

For any rectangular (non-square) contiguous matrix, the transpose IS a view (shares storage), but its strides are reversed — `(1, W)` instead of `(W, 1)` — so it's no longer contiguous in row-major order.

In [ ]:
def ex2_transpose_contig_flags(x: Tensor) -> tuple[bool, bool]:
    """Return (x.is_contiguous(), x.T.is_contiguous())."""
    raise NotImplementedError()


def _test_ex2():
    x = t.arange(12).reshape(3, 4).float()
    orig, t_contig = ex2_transpose_contig_flags(x)
    assert orig is True, f'fresh reshaped tensor should be contiguous, got {orig}'
    assert t_contig is False, f'transpose of (3,4) should be non-contiguous, got {t_contig}'
    # Sanity: storage is shared (the transpose is a view, not a copy).
    assert x.data_ptr() == x.T.data_ptr(), 'x and x.T should share storage'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_transpose_contig_flags(x: Tensor) -> tuple:
    return (x.is_contiguous(), x.T.is_contiguous())
```

**Why transpose is a view.** `.T` doesn't move data — it just swaps the strides. For `(3, 4)` with strides `(4, 1)`, the transpose has shape `(4, 3)` and strides `(1, 4)` over the SAME storage buffer. The result is logically transposed but the memory pattern no longer matches row-major contiguous layout, hence `is_contiguous() → False`.
</details>

### Exercise 3 — .view() raises on non-contiguous input

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply a try/except to detect that `.view()` raises on a non-contiguous tensor.
> Keywords: view, runtime-error, contiguity-requirement
> ```

**KCs targeted:** `view-requires-contig`

Implement `ex3_view_fails_on_transpose(x)` to return `True` if calling `.view(-1)` on `x.T` raises `RuntimeError`, `False` if it does not.

`.view()` requires the source to be contiguous. The transpose of a rectangular matrix is a view but not contiguous → `.view()` complains rather than silently moving data.

Use a `try/except RuntimeError` block.

In [ ]:
def ex3_view_fails_on_transpose(x: Tensor) -> bool:
    """Return True if x.T.view(-1) raises RuntimeError, else False."""
    raise NotImplementedError()


def _test_ex3():
    x = t.arange(12).reshape(3, 4).float()
    assert ex3_view_fails_on_transpose(x) is True, '.view(-1) on x.T should raise on non-contiguous'
    # Sanity: the original tensor is contiguous so .view(-1) on it works.
    assert x.view(-1).shape == (12,), 'view on a contiguous tensor should work'
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_view_fails_on_transpose(x: Tensor) -> bool:
    try:
        x.T.view(-1)
        return False
    except RuntimeError:
        return True
```

**Why `.view()` is strict.** It's the zero-copy reshape. It must succeed only when the new shape is compatible with the existing stride pattern — which for arbitrary reshapes means the source must be contiguous. If you want 'reshape if possible, copy if needed' use `.reshape()` instead — it falls back to a copy silently.
</details>

### Exercise 4 — fix .view() with .contiguous()

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `.contiguous()` before `.view()` to copy a non-contiguous view into a fresh row-major layout.
> Keywords: contiguous-copy, view-fix, row-major
> ```

**KCs targeted:** `contiguous-fixes-view`

Implement `ex4_flatten_transpose(x)` to flatten `x.T` to a 1-D tensor in row-major (C) order, the **transpose-then-flatten** layout.

Input shape: `(H, W)`. Output shape: `(H * W,)`. Order: read `x.T` row by row, top to bottom (so the result is `[x[0,0], x[1,0], x[2,0], ..., x[0,1], x[1,1], ...]` — column-major over the original).

**Strategy:** call `.contiguous()` on `x.T` first (materializes a fresh contiguous tensor with the transposed layout), then `.view(-1)`. Equivalent: `x.T.reshape(-1)` (PyTorch's reshape does the copy for you when needed). Both are accepted.

In [ ]:
def ex4_flatten_transpose(x: Tensor) -> Tensor:
    """Flatten x.T in row-major order. (H, W) → (H*W,)."""
    raise NotImplementedError()


def _test_ex4():
    x = t.arange(12).reshape(3, 4).float()
    y = ex4_flatten_transpose(x)
    assert y.shape == (12,), f'expected (12,), got {tuple(y.shape)}'
    # Expected order: column-major over the original = [x[0,0],x[1,0],x[2,0], x[0,1],x[1,1],x[2,1], ...]
    expected = t.tensor([0., 4., 8., 1., 5., 9., 2., 6., 10., 3., 7., 11.])
    assert t.equal(y, expected), f'value mismatch: {y.tolist()} vs {expected.tolist()}'
    _dd_passed.add('ex4')
    print("ex4 ✓")

_test_ex4()

<details><summary>Solution</summary>

```python
def ex4_flatten_transpose(x: Tensor) -> Tensor:
    return x.T.contiguous().view(-1)
```

**`.contiguous()` semantics.** If the tensor is already contiguous, it's a no-op (returns `self`). Otherwise it allocates a fresh storage buffer and copies values into row-major order. After that any view operation is legal.

**Pythonic shortcut:** `x.T.reshape(-1)`. `.reshape()` tries to produce a view (zero copy) and falls back to copy if it can't. Most code uses `.reshape()` and forgets about `.contiguous()` entirely — knowing the distinction is mostly defensive.
</details>

### Exercise 5 — rolling window via as_strided (zero-copy)

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Synthesize stride arithmetic with `as_strided` to build an O(1)-memory sliding-window view of a 1-D tensor.
> Keywords: as-strided, sliding-window, integration, multi-kc
> ```

**KCs targeted:** `strides-anatomy`, `as-strided-rolling-window`

Implement `ex5_rolling_window(x, w)` to build a 2-D view where row `i` is the length-`w` window starting at position `i` of `x`.

Input shape: `(N,)` (1-D). Output shape: `(N - w + 1, w)`. Each `out[i, j] == x[i + j]`.

Use `x.as_strided(size, stride)` directly — no Python loop, no `.clone()`, no `torch.cat`. The result must share storage with `x` (it's a view).

**Stride hint:** moving one step along the output's row axis = moving one step along the source. Moving one step along the output's column axis = also moving one step along the source. So both output strides equal `x.stride(0)`.

> ⚠️ **Integrative exercise.** This combines stride mechanics + memory aliasing + the as_strided API surface — empirical work (Lohr et al. ITiCSE 2025) shows 3-concept LLM-generated exercises drop from ~94% to ~40% solvability. Expect a step in difficulty here vs Exercises 1-4.

In [ ]:
def ex5_rolling_window(x: Tensor, w: int) -> Tensor:
    """Sliding window over a 1-D tensor. (N,) → (N-w+1, w) view."""
    raise NotImplementedError()


def _test_ex5():
    x = t.arange(7).float()                 # [0, 1, 2, 3, 4, 5, 6]
    y = ex5_rolling_window(x, w=3)
    assert y.shape == (5, 3), f'expected (5,3), got {tuple(y.shape)}'
    expected = t.tensor([
        [0., 1., 2.],
        [1., 2., 3.],
        [2., 3., 4.],
        [3., 4., 5.],
        [4., 5., 6.],
    ])
    assert t.equal(y, expected), f'value mismatch:\n{y}'
    # Must be a view, not a copy — same storage pointer.
    assert y.data_ptr() == x.data_ptr(), 'rolling-window result should share storage with x (zero-copy view)'
    _dd_passed.add('ex5')
    print("ex5 ✓")

_test_ex5()

<details><summary>Solution</summary>

```python
def ex5_rolling_window(x: Tensor, w: int) -> Tensor:
    n = x.shape[0]
    s = x.stride(0)
    return x.as_strided(size=(n - w + 1, w), stride=(s, s))
```

**Reading the stride pair.**
- Output stride for axis 0 = `s` (step from row `i` to row `i+1` = advance one element in source).
- Output stride for axis 1 = `s` (step from column `j` to column `j+1` within a row = also advance one element).

Both row and column advance one source-element. The 2-D view **overlaps** itself — every source element appears in multiple rows. That's the magic: O(1) memory, no copy.

**Danger zone.** `as_strided` does NOT check bounds. If you pass a stride/size that walks off the end of storage, you get undefined behavior — silent garbage, or a segfault. Use the higher-level `torch.nn.functional.unfold` / `torch.tensor.unfold(dim, size, step)` for sliding windows in production code; this exercise is to understand what those calls do under the hood.
</details>

## Done

Run the cell below to report your progress to Delta Drills. The beacon fires only if all 5 exercises passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1', 'ex2', 'ex3', 'ex4', 'ex5'}

def _dd_feedback_level(num_passed: int) -> str:
    """Map exercise-pass count → arena-rating feedback enum."""
    if num_passed == 5: return 'not_much'
    if num_passed >= 3: return 'somewhat'
    return 'a_lot'

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {len(missing)} exercises still failing: {sorted(missing)}.")
        print("[Delta Drills] not reporting until all 5 pass.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}',
        'subtopics': [DD_SUBTOPIC],
        'feedback': _dd_feedback_level(len(_dd_passed)),
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()